# Customer Churn: EDA and First Model

This notebook explores IBM's public sample for a fictional telecom company. The `Churn` label records whether a customer left in the previous month. We can learn to classify churn status from customer attributes, but this static file cannot predict an exact cancellation date or how many minutes/GB a customer used.

Run this notebook from the repository root or from `notebooks/`. The main application and training code live in `src/churn/`; the notebook imports those same preprocessing helpers so the experiment matches the deployed model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

here = Path.cwd()
PROJECT_ROOT = here.parent if here.name == 'notebooks' else here
sys.path.insert(0, str(PROJECT_ROOT))
from src.churn.data import load_and_clean_data
from src.churn.modeling import make_models, make_pipeline, RANDOM_STATE

sns.set_theme(style='whitegrid')
X, y, df = load_and_clean_data(PROJECT_ROOT / 'data' / 'raw' / 'Telco-Customer-Churn.csv')
print(f'Rows after duplicate/target cleaning: {len(df):,}')
print(f'Features: {X.shape[1]} | Churn rate: {y.mean():.1%}')
display(df.head())

## Data quality checks

`TotalCharges` arrives as text in the source. Blank strings are converted to missing numeric values; the model pipeline will impute them using training-fold medians. We remove duplicate rows and duplicate customer IDs, but we do not discard high/low bills just because they are unusual: they may be valid customer behavior.

In [ ]:
print('Duplicate customer IDs:', df['customerID'].duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).head(10).rename('missing values').to_frame())
display(df.dtypes.rename('dtype').to_frame())

## Churn distribution

A class imbalance means a model can get seemingly good accuracy by favoring the majority class. We'll compare recall, precision, F1, and ROC-AUC as well as accuracy. Recall answers: of customers who churned, how many did the model catch?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Churn', ax=ax, color='#2364AA')
ax.set(title='Customer outcome in the sample', xlabel='Churn (0 = stayed, 1 = left)', ylabel='Customers')
plt.tight_layout()
plt.show()
print(df['Churn'].value_counts().rename(index={0: 'Stayed', 1: 'Churned'}))
print(f'Overall sample churn rate: {df.Churn.mean():.1%}')

## Churn by contract and payment method

The bars show the observed churn fraction within each group. These are associations, not proof that the contract or payment choice caused churn. Check group sizes before using a rate to make a business decision.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
contract_rate = df.groupby('Contract', observed=False)['Churn'].mean().sort_values(ascending=False)
sns.barplot(x=contract_rate.index, y=contract_rate.values, ax=axes[0], color='#E07A5F')
axes[0].set(title='Observed churn rate by contract', xlabel='Contract', ylabel='Churn rate')
axes[0].tick_params(axis='x', rotation=20)
payment_rate = df.groupby('PaymentMethod', observed=False)['Churn'].mean().sort_values(ascending=False)
sns.barplot(x=payment_rate.index, y=payment_rate.values, ax=axes[1], color='#2364AA')
axes[1].set(title='Observed churn rate by payment method', xlabel='Payment method', ylabel='Churn rate')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()
display(pd.DataFrame({'customers': df.groupby('Contract', observed=False).size(), 'churn_rate': contract_rate}))

## Tenure and monthly charges

Compare distributions rather than only group averages. Look for overlap: even if churners tend to have shorter tenure or different charges, many individuals may still be hard to distinguish.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x='tenure', hue='Churn', bins=24, stat='density', common_norm=False,
             element='step', fill=False, ax=axes[0], palette=['#2364AA', '#E07A5F'])
axes[0].set(title='Tenure distribution by churn label', xlabel='Tenure (months)')
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1], color='#E07A5F')
axes[1].set(title='Monthly charges by churn label', xlabel='Churn (0 = stayed, 1 = left)')
plt.tight_layout()
plt.show()

## Correlations and outlier flags

Pearson correlation describes linear relationships among numeric fields. It cannot summarize categorical variables, and it does not imply causation. The IQR table flags values beyond 1.5 × IQR for review; it does not automatically remove them.

In [ ]:
numeric = df.select_dtypes(include='number')
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(numeric.corr(), cmap='vlag', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('Correlation between numeric fields')
plt.tight_layout()
plt.show()

outlier_rows = []
for column in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    flagged = ((df[column] < q1 - 1.5 * iqr) | (df[column] > q3 + 1.5 * iqr)).sum()
    outlier_rows.append({'feature': column, 'IQR-flagged rows': int(flagged)})
display(pd.DataFrame(outlier_rows))

## First leakage-safe experiment

The stratified split keeps the churn proportion similar across train and test. The full preprocessing pipeline is cross-validated, so imputers, scaling, and category encoding are learned only from each training fold. Class weighting gives greater penalty to mistakes on the minority churn class.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = {}
for name in ['Logistic Regression', 'Random Forest']:
    pipeline = make_pipeline(X_train, make_models()[name])
    result = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], n_jobs=1
    )
    scores[name] = {metric: result[f'test_{metric}'].mean() for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
display(pd.DataFrame(scores).T.sort_values('f1', ascending=False).round(3))

# Fit a simple baseline pipeline and inspect one untouched test split.
baseline = make_pipeline(X_train, make_models()['Logistic Regression'])
baseline.fit(X_train, y_train)
test_probability = baseline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)
print('Test recall:', round(recall_score(y_test, test_prediction), 3))
print('Test precision:', round(precision_score(y_test, test_prediction, zero_division=0), 3))
print('Test F1:', round(f1_score(y_test, test_prediction), 3))
print('Test ROC-AUC:', round(roc_auc_score(y_test, test_probability), 3))
print(classification_report(y_test, test_prediction, target_names=['Stayed', 'Churned'], zero_division=0))
print('Confusion matrix (rows=actual, columns=predicted):\n', confusion_matrix(y_test, test_prediction))

## What to take away

1. Read the actual chart values before making claims about customer groups.
2. Keep the test split untouched until model choice and threshold selection are complete. The full `python train.py` workflow does this and tunes a threshold from training-only out-of-fold predictions.
3. Recall matters when missing a likely churner is costly; precision matters because retention outreach also has a cost. Select a threshold with business capacity in mind.
4. Service subscription fields indicate which services a customer has, not product usage volume. To answer 'what product is used most' in usage terms, collect monthly minutes, data, streaming, or transaction counts.
5. These benchmark metrics are not a guarantee of future company performance. Validate on recent, representative data before operational use.